# Reintegration readiness (Hiraya Haven)

**Pipeline:** Predict reintegration status/readiness and analyze correlates of successful progress.

**Data:** `residents`, `home_visitations`, `intervention_plans`, `education_records`, `health_wellbeing_records`, `incident_reports` from `../Data/hiraya.db`.

**Predictive:** classify `reintegration_status` (e.g., Completed vs not) using *pre-status* aggregate features.

**Explanatory:** interpretable associations between program engagement indicators and reintegration outcomes.

> Ethics note: Use as a decision-support tool, not an automated decision maker.

> **IS455 pipeline structure:** (1) Problem framing → (2) Data acquisition, preparation & exploration → (3) Modeling → (4) Evaluation & selection → (5) Feature selection & interpretation → (6) Explanatory analysis (associations; not causal proof) → (7) Deployment.

## 1. Problem framing

Leadership needs to understand which residents may be ready for reintegration and which might require additional support (`IntexContext.txt`).

**Business question:** *Which residents are likely to reach `reintegration_status = Completed`, and what factors correlate with successful completion?*

- **Predictive goal:** classify completion vs not completed.
- **Explanatory goal:** interpret correlates (visitation patterns, interventions, education/health trends, incident history).

**Success metrics:** ROC-AUC / AP; and clear, honest interpretation of limitations.

## 2. Data acquisition, preparation & exploration

We load tables from `../Data/hiraya.db` and create **resident-level aggregates** for visits, intervention plans, education/health trajectories, and incident history.

**Leakage control:** In production, features should use only information available prior to the prediction point (e.g., last 6 months before a review date). This notebook uses overall aggregates as a baseline.

Downstream code cells implement **§3 Modeling**, **§4 Evaluation & selection**, and the add-on for stricter time logic.

In [5]:
# ============================================================================
# Reintegration readiness — Load tables and define completion target
# ============================================================================
# Notebook code cell overview — see inline comments below.
#
# Binary target_completed when reintegration_status == 'Completed' (adjust if business rules change).
# Sensitive: resident-level outcomes; keep scores staff-only in production.
#
from __future__ import annotations

import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

sns.set_theme(style="whitegrid", context="notebook")

ROOT = Path("..").resolve()
DB_PATH = ROOT / "Data" / "hiraya.db"

if not DB_PATH.exists():
    raise FileNotFoundError(f"Database not found at {DB_PATH}")


def read_table(conn: sqlite3.Connection, table: str) -> pd.DataFrame:
    return pd.read_sql_query(f'SELECT * FROM "{table}"', conn)


with sqlite3.connect(DB_PATH) as conn:
    residents = read_table(conn, "residents")
    visits = read_table(conn, "home_visitations")
    plans = read_table(conn, "intervention_plans")
    edu = read_table(conn, "education_records")
    health = read_table(conn, "health_wellbeing_records")
    incidents = read_table(conn, "incident_reports")

visits["visit_date"] = pd.to_datetime(visits["visit_date"], errors="coerce")
plans["target_date"] = pd.to_datetime(plans["target_date"], errors="coerce")
edu["record_date"] = pd.to_datetime(edu["record_date"], errors="coerce")
health["record_date"] = pd.to_datetime(health["record_date"], errors="coerce")
incidents["incident_date"] = pd.to_datetime(incidents["incident_date"], errors="coerce")

# Target (binary)
res = residents.copy()
res["target_completed"] = (res["reintegration_status"].fillna("") == "Completed").astype(int)
res[["reintegration_status", "target_completed"]].value_counts().head(10)

reintegration_status  target_completed
In Progress           0                   21
Completed             1                   19
On Hold               0                   13
Not Started           0                    7
Name: count, dtype: int64

In [6]:
# ============================================================================
# Feature engineering — Resident-level aggregates from visits, plans, edu, health, incidents
# ============================================================================
# Notebook code cell overview — see inline comments below.
#
# Sums and means summarize trajectory; last_* proxies 'most recent' progress/health when available.
# Missing numeric aggregates are filled after train/test split using training medians only (Ch. 15).
#
# Feature engineering: resident-level aggregates (keep it simple & privacy-safe)

vis_agg = (
    visits.dropna(subset=["visit_date"]).assign(month=visits["visit_date"].dt.to_period("M").dt.to_timestamp())
    .groupby("resident_id", as_index=False)
    .agg(
        visits_total=("visitation_id", "count"),
        follow_up_total=("follow_up_needed", lambda s: int((s == 1).sum())),
        safety_concerns_total=("safety_concerns_noted", lambda s: int((s == 1).sum())),
        unfavorable_total=("visit_outcome", lambda s: int((s == "Unfavorable").sum())),
    )
)

plan_agg = (
    plans.groupby("resident_id", as_index=False)
    .agg(
        plans_total=("plan_id", "count"),
        open_plans=("status", lambda s: int((s.isin(["Open", "In Progress"])).sum())),
    )
)

edu_agg = (
    edu.groupby("resident_id", as_index=False)
    .agg(
        edu_records=("education_record_id", "count"),
        mean_attendance=("attendance_rate", "mean"),
        mean_progress=("progress_percent", "mean"),
        last_progress=("progress_percent", lambda s: float(pd.Series(s.dropna()).iloc[-1]) if s.dropna().size else np.nan),
    )
)

health_agg = (
    health.groupby("resident_id", as_index=False)
    .agg(
        health_records=("health_record_id", "count"),
        mean_health=("general_health_score", "mean"),
        last_health=("general_health_score", lambda s: float(pd.Series(s.dropna()).iloc[-1]) if s.dropna().size else np.nan),
    )
)

inc_agg = (
    incidents.groupby("resident_id", as_index=False)
    .agg(
        incidents_total=("incident_id", "count"),
        high_sev_total=("severity", lambda s: int((s == "High").sum())),
        unresolved_total=("resolved", lambda s: int((s == 0).sum())),
    )
)

df = (
    res[[
        "resident_id",
        "safehouse_id",
        "case_status",
        "case_category",
        "initial_risk_level",
        "current_risk_level",
        "reintegration_type",
        "target_completed",
    ]]
    .merge(vis_agg, on="resident_id", how="left")
    .merge(plan_agg, on="resident_id", how="left")
    .merge(edu_agg, on="resident_id", how="left")
    .merge(health_agg, on="resident_id", how="left")
    .merge(inc_agg, on="resident_id", how="left")
)

for c in df.columns:
    if c.endswith("_total") or c.endswith("_records") or c in ["plans_total", "open_plans"]:
        df[c] = df[c].fillna(0).astype(int)

df[["target_completed"]].mean()

target_completed    0.316667
dtype: float64

In [7]:
# ============================================================================
# Model — Logistic regression with grouped categorical + scaled numeric features
# ============================================================================
# Notebook code cell overview — see inline comments below.
#
# Random stratified split is convenient but can leak across time; see leakage-reduced cell 8.
# ROC-AUC / AP summarize discrimination; class_weight='balanced' helps rare positive class.
#
# Modeling

cat_cols = [
    "case_status",
    "case_category",
    "initial_risk_level",
    "current_risk_level",
    "reintegration_type",
]
num_cols = [
    "visits_total",
    "follow_up_total",
    "safety_concerns_total",
    "unfavorable_total",
    "plans_total",
    "open_plans",
    "edu_records",
    "mean_attendance",
    "mean_progress",
    "health_records",
    "mean_health",
    "incidents_total",
    "high_sev_total",
    "unresolved_total",
]

X = df[cat_cols + num_cols]
y = df["target_completed"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

for _c in ["mean_attendance", "mean_progress", "mean_health"]:
    _m = X_train[_c].median()
    X_train[_c] = X_train[_c].fillna(_m)
    X_test[_c] = X_test[_c].fillna(_m)

pre = ColumnTransformer(
    [
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", Pipeline([("scaler", StandardScaler())]), num_cols),
    ]
)

pipe = Pipeline(
    [
        ("pre", pre),
        (
            "lr",
            LogisticRegression(max_iter=500, class_weight="balanced", solver="liblinear"),
        ),
    ]
)

pipe.fit(X_train, y_train)
proba = pipe.predict_proba(X_test)[:, 1]
roc = roc_auc_score(y_test, proba) if y_test.nunique() > 1 else float("nan")
ap = average_precision_score(y_test, proba)
print(f"ROC-AUC={roc:.3f} AP={ap:.3f}")

ROC-AUC=0.520 AP=0.406


## 3. Modeling

Logistic regression with one-hot encoded categorical resident attributes and scaled numeric aggregates.

## 4. Evaluation & selection

We evaluate using ROC-AUC and Average Precision (AP). Interpret results as **ranking quality** for prioritizing staff review—not as an automated decision.

Discuss consequences:
- **False negatives**: a resident who needs support is deprioritized.
- **False positives**: staff time is spent reviewing a resident who would have progressed anyway.

## 5. Feature selection & interpretation

Use **model coefficients** (or stepwise removal experiments) to discuss which aggregates matter; keep narrative aligned with domain: visits/plans can be **responses** to risk, not just drivers.

## 6. Explanatory analysis (associations; not causal proof)

Aggregates like visits and plans can be responses to risk (reverse causation). Treat coefficients/feature effects as **associations**, not proof that more/less visits cause completion.

## 7. Deployment

- Add a “Reintegration readiness” view with predicted completion probability (or readiness score).
- Store scores per resident with a model version.
- Monitor drift and fairness (e.g., avoid systematically deprioritizing certain case categories).

## Add-on: leakage-reduced snapshot logic + out-of-time evaluation

The baseline model above uses full-history aggregates. A more appropriate logic is to build features using data only up to a **snapshot date**.

- For residents with `date_closed`, use that as the snapshot.
- Otherwise use the last observed record date across education/health/visits/incidents.

Then evaluate with an **out-of-time split** by snapshot date.

In [8]:
# ============================================================================
# Leakage-reduced evaluation — Snapshot date and out-of-time split
# ============================================================================
# Notebook code cell overview — see inline comments below.
#
# Uses date_closed or last activity to define what data could have been known at prediction time.
# Filters visits/edu/health/incidents to on-or-before snapshot for each resident.
# Retrains the same pipeline on train mask only; tests on later snapshot_dates.
#
res2 = residents.copy()
res2["date_closed"] = pd.to_datetime(res2["date_closed"], errors="coerce")

# Recreate target on res2 (fixes KeyError)
res2["target_completed"] = (res2["reintegration_status"].fillna("") == "Completed").astype(int)

# Per-resident last observed date across sources
last_dates = []
for name, d, col in [
    ("visits", visits, "visit_date"),
    ("edu", edu, "record_date"),
    ("health", health, "record_date"),
    ("incidents", incidents, "incident_date"),
]:
    tmp = d.dropna(subset=[col]).groupby("resident_id", as_index=False)[col].max().rename(columns={col: f"last_{name}"})
    last_dates.append(tmp)

last = last_dates[0]
for t in last_dates[1:]:
    last = last.merge(t, on="resident_id", how="outer")

res2 = res2.merge(last, on="resident_id", how="left")
res2["snapshot_date"] = res2["date_closed"]

# If no date_closed, use max of last_* dates
max_last = res2[[c for c in res2.columns if c.startswith("last_")]].max(axis=1)
res2.loc[res2["snapshot_date"].isna(), "snapshot_date"] = max_last

# Filter event tables to <= snapshot_date
vis_f = visits.merge(res2[["resident_id", "snapshot_date"]], on="resident_id", how="left")
vis_f = vis_f[vis_f["visit_date"] <= vis_f["snapshot_date"]]

edu_f = edu.merge(res2[["resident_id", "snapshot_date"]], on="resident_id", how="left")
edu_f = edu_f[edu_f["record_date"] <= edu_f["snapshot_date"]]

health_f = health.merge(res2[["resident_id", "snapshot_date"]], on="resident_id", how="left")
health_f = health_f[health_f["record_date"] <= health_f["snapshot_date"]]

inc_f = incidents.merge(res2[["resident_id", "snapshot_date"]], on="resident_id", how="left")
inc_f = inc_f[inc_f["incident_date"] <= inc_f["snapshot_date"]]

# Rebuild aggregates using filtered data
vis_agg2 = (
    vis_f.groupby("resident_id", as_index=False)
    .agg(
        visits_total=("visitation_id", "count"),
        follow_up_total=("follow_up_needed", lambda s: int((s == 1).sum())),
        safety_concerns_total=("safety_concerns_noted", lambda s: int((s == 1).sum())),
        unfavorable_total=("visit_outcome", lambda s: int((s == "Unfavorable").sum())),
    )
)

edu_agg2 = (
    edu_f.groupby("resident_id", as_index=False)
    .agg(
        edu_records=("education_record_id", "count"),
        mean_attendance=("attendance_rate", "mean"),
        mean_progress=("progress_percent", "mean"),
        last_progress=("progress_percent", lambda s: float(pd.Series(s.dropna()).iloc[-1]) if s.dropna().size else np.nan),
    )
)

health_agg2 = (
    health_f.groupby("resident_id", as_index=False)
    .agg(
        health_records=("health_record_id", "count"),
        mean_health=("general_health_score", "mean"),
        last_health=("general_health_score", lambda s: float(pd.Series(s.dropna()).iloc[-1]) if s.dropna().size else np.nan),
    )
)

inc_agg2 = (
    inc_f.groupby("resident_id", as_index=False)
    .agg(
        incidents_total=("incident_id", "count"),
        high_sev_total=("severity", lambda s: int((s == "High").sum())),
        unresolved_total=("resolved", lambda s: int((s == 0).sum())),
    )
)

plans["created_at"] = pd.to_datetime(plans["created_at"], errors="coerce")
plans_snap = res2[["resident_id", "snapshot_date"]].merge(plans, on="resident_id", how="left")
plans_snap = plans_snap[plans_snap["created_at"].notna() & (plans_snap["created_at"] <= plans_snap["snapshot_date"])]
plan_agg2 = (
    plans_snap.groupby("resident_id", as_index=False)
    .agg(
        plans_total=("plan_id", "count"),
        open_plans=("status", lambda s: int((s.isin(["Open", "In Progress"])).sum())),
    )
)

df2 = (
    res2[[
        "resident_id",
        "snapshot_date",
        "case_status",
        "case_category",
        "initial_risk_level",
        "current_risk_level",
        "reintegration_type",
        "target_completed",
    ]]
    .merge(vis_agg2, on="resident_id", how="left")
    .merge(plan_agg2, on="resident_id", how="left")
    .merge(edu_agg2, on="resident_id", how="left")
    .merge(health_agg2, on="resident_id", how="left")
    .merge(inc_agg2, on="resident_id", how="left")
)

for c in [
    "visits_total",
    "follow_up_total",
    "safety_concerns_total",
    "unfavorable_total",
    "plans_total",
    "open_plans",
    "edu_records",
    "health_records",
    "incidents_total",
    "high_sev_total",
    "unresolved_total",
]:
    df2[c] = df2[c].fillna(0).astype(int)

# Out-of-time split by snapshot_date
cut = df2["snapshot_date"].dropna().quantile(0.75)
train = df2["snapshot_date"] < cut

for _c in ["mean_attendance", "mean_progress", "mean_health"]:
    _m = df2.loc[train, _c].median()
    df2[_c] = df2[_c].fillna(_m)

X2 = df2[cat_cols + num_cols]
y2 = df2["target_completed"].astype(int)

pipe.fit(X2[train], y2[train])
proba = pipe.predict_proba(X2[~train])[:, 1]
roc2 = roc_auc_score(y2[~train], proba) if y2[~train].nunique() > 1 else float("nan")
ap2 = average_precision_score(y2[~train], proba)
print(f"Leakage-reduced out-of-time: ROC-AUC={roc2:.3f} AP={ap2:.3f}")

Leakage-reduced out-of-time: ROC-AUC=0.682 AP=0.402
